# 01 — Data exploration
**Primorsk oil terminal, Gulf of Finland**

What data do we have before building anything? Two streams:
1. **Sentinel-1 GRD** scenes (live from Planetary Computer STAC)
2. **AIS vessel presence** (the committed seed in `data/seed/`), so we avoid having to sign-up and create a token. But, you can get yours at: https://globalfishingwatch.org/our-apis/

This notebook is only for exploratory purposes.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

from pipeline.config import PORT_BBOX, DATE_START, DATE_END, COVERAGE_THRESHOLD, EVENTS
from pipeline.data import list_scenes

print('bbox', PORT_BBOX)
print('window', DATE_START, '->', DATE_END)

## 1 · Sentinel-1 scenes

In [ ]:
scenes = list_scenes(*PORT_BBOX, DATE_START, DATE_END)
df = pd.DataFrame(scenes)
df['datetime'] = pd.to_datetime(df['datetime'])
print(f'{len(df)} IW/VV scenes intersecting the port')
print('platforms:', df['platform'].value_counts().to_dict())
print('orbits:', sorted(df['relative_orbit'].dropna().unique()))
df[['scene_id','datetime','platform','relative_orbit','coverage']].head()

## 2 · Coverage

We only keep scenes that cover the port well. Orbits 7 and 14 see the whole harbour; 80 and 87 only graze it on some passes.

In [ ]:
kept = df[df['coverage'] >= COVERAGE_THRESHOLD]
print(f'cover >= {COVERAGE_THRESHOLD:.0%}: {len(kept)} of {len(df)}')
frac = (df.assign(ok=df['coverage'] >= COVERAGE_THRESHOLD)
          .groupby('relative_orbit')['ok'].mean().round(2))
print('\nfraction of each orbit covering the port:')
print(frac.to_string())

In [ ]:
df['month'] = df['datetime'].dt.to_period('M').dt.to_timestamp()
monthly = df.groupby(['month','platform']).size().unstack(fill_value=0)
ax = monthly.plot.bar(stacked=True, figsize=(13,4), width=0.85)
ax.set_title('Sentinel-1 scenes per month by platform'); ax.set_xlabel(''); ax.set_ylabel('scenes')
plt.tight_layout(); plt.show()

## 3 · AIS vessel presence (seed)

Daily unique-vessel presence inside the port, the independent signal we compare detections against. No ground truth, so this is a proxy that will help us ground our vessel identifications.

In [ ]:
ais = pd.read_parquet('../data/seed/ais_primorsk.parquet')
ais['date'] = pd.to_datetime(ais['date'])
print(f'{len(ais)} vessel-day rows; {ais.vesselId.nunique()} unique vessels')
print('top flags:', ais['flag'].value_counts().head(6).to_dict())
monthly_ais = ais.set_index('date').resample('MS')['vesselId'].nunique()
ax = monthly_ais.plot(figsize=(13,4), marker='o')
ax.set_title('AIS unique vessels per month'); ax.set_ylabel('vessels')
for d,l in EVENTS:
    ax.axvline(pd.to_datetime(d), color='gray', ls='--', alpha=0.6)
plt.tight_layout(); plt.show()

## Takeaways

- The scene supply is uneven (orbit coverage, S1B leaving the IW record in late 2021).
- AIS gives an independent count to compare against, but it is a proxy, not truth.
- Next: build a detector and register it as a model (notebook 02).